# GF4 CUDA Kernels on Colab

Builds and runs the hand-written CUDA kernels in `CUDA_FP4_Test` (Hadamard / GF4 encode-decode / Hessian weight-quant / fused E2M1 GEMV) on a Colab GPU, plus profiling.

**What this is:** the real `.cu` kernels + `bench.py` (correctness + fused-vs-naive timing) and `llm_quant_eval.py` (full pipeline on a real model). The eval path is *fake-quant + fp16 GEMM* — it measures accuracy, not inference speed. The kernel timings in `bench.py` are the actual kernel-speed evidence.

**Runtime:** set **Runtime → Change runtime type → GPU** (T4/L4/A100/H100 all work). Bigger GPU only matters for `llm_quant_eval.py` on larger models.

## 0. GPU + toolchain check (and pin the compile arch)

In [ ]:
!nvidia-smi
!nvcc --version
import os, torch
cap = torch.cuda.get_device_capability()
arch = f"{cap[0]}.{cap[1]}"   # 7.5 T4 | 8.0 A100 | 8.9 L4 | 9.0 H100
os.environ["TORCH_CUDA_ARCH_LIST"] = arch    # pin so the JIT build compiles for THIS GPU only (faster, avoids arch warnings)
print("torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| GPU", torch.cuda.get_device_name(0), "| sm_", arch.replace('.',''))

## 1. Get the code onto Colab (git clone)

`CUDA_FP4_Test` is now in **`Thesis_Compression`**, so just clone it (shallow, to stay light). The same clone also brings `iso_energy_125m.py` + the `run_offload_*.sh` scripts if you want the big-model runs later.

If the repo is **private**, add a GitHub token (PAT) as a Colab Secret named `GH_TOKEN` (🔑 in the left sidebar, and enable it for this notebook) — the cell picks it up automatically. If it's public, leave `GH_TOKEN` unset.

In [ ]:
import os
# Private repo? add a GitHub PAT as Colab Secret 'GH_TOKEN'. Public? leave it unset.
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = os.environ.get('GH_TOKEN', '')
# Build the URL in an env var so the token never gets echoed into the cell output.
os.environ['REPO_URL'] = f"https://{tok + '@' if tok else ''}github.com/jvap2/Thesis_Compression.git"
!rm -rf /content/Thesis_Compression
!git clone --depth 1 "$REPO_URL" /content/Thesis_Compression
%cd /content/Thesis_Compression/Python_Jenks_Test/Jenks_Tests/CUDA_FP4_Test
!ls *.cu *.cpp *.py

In [ ]:
# --- Alternatives (only if you can't clone) ---
# OPTION B: upload a tarball (build locally: cd CUDA_FP4_Test && tar czf cuda_fp4_test.tar.gz *.cu *.cuh *.cpp *.py README.md)
# from google.colab import files; files.upload()          # choose cuda_fp4_test.tar.gz
# !mkdir -p /content/CUDA_FP4_Test && tar xzf cuda_fp4_test.tar.gz -C /content/CUDA_FP4_Test
# %cd /content/CUDA_FP4_Test
#
# OPTION C: Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/CUDA_FP4_Test" /content/ && cd /content/CUDA_FP4_Test

In [ ]:
# Build deps. torch is preinstalled on Colab; ninja speeds the JIT build.
!pip -q install ninja
# transformers/datasets/accelerate are only needed for llm_quant_eval.py (section 3):
!pip -q install "transformers==4.46.3" datasets accelerate

## 2. Build + run the kernels (`bench.py`)

First run JIT-compiles all `.cu` into the `gf4_kernels` extension (1–3 min), then runs every kernel's correctness check and the fused-vs-naive timing races. This is the CUDA work to examine — the timing lines are the real kernel-speed numbers.

The build is cached under `~/.cache/torch_extensions`, so re-runs are instant unless the runtime resets.

In [ ]:
!TORCH_CUDA_ARCH_LIST={arch} python3 bench.py

## 3. Full pipeline on a real model (`llm_quant_eval.py`)

Runs the W4A16 / W4A4 / W4A16-Hadamard configs using the compiled CUDA kernels. Two ways to run it:

- **`--config all`** (below, on OPT-125M) — every config in ONE process. Keeps the ~model-sized fp16 clone, so it caps near ~7B. Good for the smoke test.
- **`--config <one>`** (Section 3b) — one config PER process, **no clone**, only the needed basis calibrated. This is how big models (OPT-30B) fit your A100. Each run appends to `eval_results.csv`.

**HF auth:** `llm_quant_eval.py` reads `HF_TOKEN` from the environment (no hardcoded token). `opt-*` models are ungated and need none; for gated Llama set it from a Colab Secret:

```python
from google.colab import userdata; import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
```

In [ ]:
# Smoke test: full ablation on OPT-125M in one process (--config all).
!TORCH_CUDA_ARCH_LIST={arch} python3 llm_quant_eval.py --model facebook/opt-125m \
    --config all --num-calib-windows 2 --num-eval-windows 5

### 3b. Big models, per-config — OPT-30B (no clone → fits your A100)

Each `--config` runs as its **own process**, so there's no `original_weights` clone and only the needed basis is calibrated. Every run appends a row to `eval_results.csv`.

⚠️ **Re-run the Section 1 clone cell first** to pick up the per-config code, then run these top-to-bottom. OPT is ungated — no HF token needed. Cheap configs (baseline/naive) run first; `w4a16_raw` and `rotated` use the CUDA Hessian calibration.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL  = "facebook/opt-30b"
MAXMEM = "0=38GiB,cpu=70GiB"     # A100-40GB: ~38 GB on GPU, the rest streamed to CPU RAM
COMMON = f"--model {MODEL} --device-map --max-memory {MAXMEM} --num-eval-windows 20"

# baseline + naive need NO calibration and NO clone - cheapest, run first:
!python3 llm_quant_eval.py {COMMON} --config baseline
!python3 llm_quant_eval.py {COMMON} --config naive

In [ ]:
# RAW W4A16: Hessian weight-quant in the un-rotated basis (CUDA). Raw calibration only.
!python3 llm_quant_eval.py {COMMON} --config w4a16_raw --num-calib-windows 2

In [ ]:
# ROTATED: W4A16-Hadamard + W4A4 together (shared rotated calibration, CUDA Hessian).
# Heaviest config here - it holds ~40 GB of dense rotated weights in CPU RAM.
# If it OOMs, drop to --num-calib-windows 1.
!python3 llm_quant_eval.py {COMMON} --config rotated --num-calib-windows 2

In [ ]:
# Read back every perplexity collected across the per-config runs above.
import pandas as pd
df = pd.read_csv("eval_results.csv")
print(df[df.model == MODEL].to_string(index=False))
from google.colab import files; files.download("eval_results.csv")

### 3c. OPT-66B — probably won't fit here (honest note)

OPT-66B is **~132 GB** fp16. Your A100 (40 GB) + 83.5 GB RAM ≈ 123 GB total — the model doesn't fit even to *load*, so the **quant configs can't run** on this box via this CUDA script.

- **baseline only** *might* run via disk offload (`--offload-folder`) after a ~132 GB download — very slow (cell below).
- For 66B **quant** numbers, use the memory-streaming offload path (`Offload_BigModels_Colab.ipynb` / `iso_energy_125m.py`) on a **192 GB-RAM** machine. That path streams one block at a time and needs no clone; it's the right tool for 66B, not this CUDA script.

In [ ]:
# LONG SHOT: OPT-66B baseline via disk offload. ~132 GB download first, then very
# slow streamed eval. Quant configs will NOT fit this box (see note above).
!python3 llm_quant_eval.py --model facebook/opt-66b --device-map \
    --max-memory "0=38GiB,cpu=70GiB" --offload-folder /content/offload66b \
    --config baseline --num-eval-windows 5

## 4. Profiling

| Tool | What it gives | On Colab? |
|---|---|---|
| **Nsight Systems (`nsys`)** | Timeline trace: kernel launches, durations, memcpy, gaps, overlap | ✅ usually works (CUPTI tracing) |
| **Nsight Compute (`ncu`)** | Per-kernel metrics: occupancy, memory throughput, roofline | ❌ usually **blocked** (`ERR_NVGPUCTRPERM` — needs GPU perf-counter/admin perms Colab doesn't grant) |
| **`torch.profiler`** | Per-op/kernel CUDA times, Chrome trace | ✅ works, pure-Python |

**Bottom line:** do timeline profiling with `nsys` here; keep the detailed `ncu` kernel analysis (your `profile_hbm.py` / `ncu_hbm.csv`) on the desktop where you have counter permissions. `torch.profiler` is the Colab-native fallback for per-kernel times.

### 4a. Nsight Systems (timeline trace)

In [ ]:
# nsys is NOT preinstalled on Colab (only the CUDA runtime is). Install it, or fall back to 4c.
import shutil, os, glob
def find_nsys():
    p = shutil.which("nsys")
    if p: return p
    for c in glob.glob("/opt/nvidia/nsight-systems/*/target-linux-x64/nsys") + \
             glob.glob("/opt/nvidia/nsight-systems/*/host-linux-x64/nsys"):
        return c
    return None
nsys = find_nsys()
if not nsys:
    print("installing nsight-systems-cli via apt...")
    !apt-get -qq update && apt-get -qq install -y nsight-systems-cli 2>&1 | tail -3
    nsys = find_nsys()
if nsys:
    os.environ["PATH"] = os.path.dirname(nsys) + os.pathsep + os.environ["PATH"]
    print("nsys ready:", nsys)
    !nsys --version
else:
    print("nsys unavailable on this image — skip 4a and use the torch.profiler cell (4c); it works on Colab.")

In [ ]:
# Trace bench.py. --trace uses CUPTI (allowed on Colab); produces gf4_bench.nsys-rep.
!TORCH_CUDA_ARCH_LIST={arch} nsys profile \
    --trace=cuda,nvtx,osrt --force-overwrite=true \
    -o gf4_bench python3 bench.py
print("\n--- report summary ---")
!nsys stats --report gpukernsum gf4_bench.nsys-rep 2>/dev/null | head -40

In [ ]:
# Download the .nsys-rep and open it in the Nsight Systems GUI on your machine.
from google.colab import files
files.download("gf4_bench.nsys-rep")

### 4b. Nsight Compute (expected to fail on Colab — shown for the record)

In [ ]:
# Expect: ==ERROR== ERR_NVGPUCTRPERM ... GPU Performance Counters.
# This is a Colab permission limit, not a bug in the kernels. Run ncu on the desktop.
!which ncu && ncu --version || echo "ncu not present"
!ncu --set basic --launch-count 3 -f -o gf4_ncu \
    python3 -c "import torch; from torch.utils.cpp_extension import load" 2>&1 | head -15

### 4c. `torch.profiler` (Colab-native per-kernel timing)

Profiles a representative loop of the GF4 encode/decode + Hadamard kernels in-process. Prints the CUDA-time table and writes a Chrome trace (`chrome://tracing` or `edge://tracing` to view).

In [ ]:
import torch
from torch.utils.cpp_extension import load
from torch.profiler import profile, ProfilerActivity

ext = load(name="gf4_kernels", sources=[
    "bindings.cpp", "hadamard_kernel.cu", "gf4_encode_kernel.cu",
    "hessian_weight_quant_kernel.cu", "e2m1_fused_gemv_kernel.cu"],
    extra_cuda_cflags=["-O3", "--use_fast_math"], verbose=False)

# Representative activation tensor + a random-sign Hadamard vector.
HAD_BLOCK, GF4_BLOCK, CLIP = 32, 32, 2.5
x = torch.randn(8192, 4096, device="cuda").reshape(-1, HAD_BLOCK).contiguous()
d_sign = (torch.randint(0, 2, (HAD_BLOCK,), device="cuda") * 2 - 1).float()
mu = torch.zeros(HAD_BLOCK, device="cuda")

def step():
    xh = ext.hadamard_fwht(x, HAD_BLOCK, d_sign)
    nb = xh.numel() // GF4_BLOCK
    codes, scales = ext.gf4_encode(xh.reshape(-1).contiguous(), CLIP, True, mu)
    xq = ext.gf4_decode(codes, scales, nb)
    torch.cuda.synchronize()

for _ in range(3):
    step()   # warm up (JIT, allocator)

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    for _ in range(20):
        step()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
prof.export_chrome_trace("gf4_trace.json")
from google.colab import files; files.download("gf4_trace.json")